In [1]:
import yaml
import pickle

In [2]:
config_file = "../config_multiscope/podns_multi_scale5k10k.yaml"
with open(config_file, 'r') as file:
    config = yaml.safe_load(file)

In [3]:
p_filename = "../" + config['experiment_name'] + "_ts.pkl"

with open(p_filename, 'rb') as file:
    src = pickle.load(file)
    dst = pickle.load(file)

In [4]:
dst['/tordata/config/group_0_user_0']

,text_len,count
time,,
2025-06-20 13:23:48.510,86,1
2025-06-20 13:23:51.843,0,0
2025-06-20 13:23:55.176,163,1
2025-06-20 13:23:58.509,0,0
2025-06-20 13:24:01.842,0,0
...,...,...
2025-06-20 19:15:59.730,0,0
2025-06-20 19:16:03.063,0,0
2025-06-20 19:16:06.396,0,0


In [5]:
src['102.0.0.10']

,ip.proto,ip.len,tcp.dstport,tcp.ack,tcp.len,tcp.reassembled.length,udp.dstport,tcp.time_relative,tcp.time_delta,count,...,count_Service-2,ip.proto_Service-3,ip.len_Service-3,tcp.dstport_Service-3,tcp.ack_Service-3,tcp.len_Service-3,tcp.time_relative_Service-3,tcp.time_delta_Service-3,count_Service-3,frame.time
frame.time,,,,,,,,,,,,,,,,,,,,,
2025-06-20 11:59:49.014,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,3.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.00000,0.0,2025-06-20 11:59:49.014
2025-06-20 11:59:52.347,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.00000,0.0,2025-06-20 11:59:52.347
2025-06-20 11:59:55.680,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.00000,0.0,2025-06-20 11:59:55.680
2025-06-20 11:59:59.013,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.00000,0.0,2025-06-20 11:59:59.013
2025-06-20 12:00:02.346,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.00000,0.0,2025-06-20 12:00:02.346
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-06-20 19:15:59.730,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.00000,0.0,2025-06-20 19:15:59.730
2025-06-20 19:16:03.063,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.00000,0.0,2025-06-20 19:16:03.063
2025-06-20 19:16:06.396,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.00000,0.0,2025-06-20 19:16:06.396


In [6]:
import pandas as pd 

def ts2nccvec(ts: pd.core.frame.DataFrame):
    sample_offset = ((len(ts)-1)**0.5)
    return (ts - ts.mean()) / (ts.std()*sample_offset)
    
vec = ts2nccvec(dst['/tordata/config/group_0_user_0']["text_len"])
vec

time
2025-06-20 13:23:48.510    0.021954
2025-06-20 13:23:51.843   -0.000851
2025-06-20 13:23:55.176    0.042373
2025-06-20 13:23:58.509   -0.000851
2025-06-20 13:24:01.842   -0.000851
                             ...   
2025-06-20 19:15:59.730   -0.000851
2025-06-20 19:16:03.063   -0.000851
2025-06-20 19:16:06.396   -0.000851
2025-06-20 19:16:09.729   -0.000851
2025-06-20 19:16:13.062    0.051124
Name: text_len, Length: 6345, dtype: float64

In [7]:
import numpy as np
np.dot(vec.to_numpy(), vec.to_numpy())

np.float64(0.9999999999999981)

### Todo
- **Find way to align dataset**
    - chunk, fill zeros, etc
    - _Last big thing to do before all out test of method_
- Get end to end working
    - Send pcaps dfs through ncc GIS stuff
- Run new experiment loop
    - run through all features/combos and report metrics

In [8]:
scale_factor = 300
chunk_size = td = pd.Timedelta(1, "W")/scale_factor
chunk_size

Timedelta('0 days 00:33:36')

In [9]:
start = min([src[ip].index[0] for ip in src])
start

Timestamp('2025-06-20 11:59:49.014000')

In [10]:
# Fill zeros in missing indexes
full_index = pd.Index([])

for ip in src:
    full_index = full_index.union(src[ip].index)
    
for user in dst:
    full_index = full_index.union(dst[user].index)
print(full_index)
for ip in src:
    src[ip] = src[ip].reindex(full_index, fill_value=0)
    assert len(src[ip]) == len(full_index)

for user in dst:
    dst[user] = dst[user].reindex(full_index, fill_value=0)
    assert len(dst[user]) == len(full_index)

Index([2025-06-20 11:59:49.014000, 2025-06-20 11:59:52.347000,
       2025-06-20 11:59:55.680000, 2025-06-20 11:59:59.013000,
       2025-06-20 12:00:02.346000, 2025-06-20 12:00:05.679000,
       2025-06-20 12:00:09.012000, 2025-06-20 12:00:12.345000,
       2025-06-20 12:00:15.678000, 2025-06-20 12:00:19.011000,
       ...
       2025-06-20 22:23:11.940000, 2025-06-20 22:23:15.273000,
       2025-06-20 22:23:18.606000, 2025-06-20 22:23:21.939000,
       2025-06-20 22:23:25.272000, 2025-06-20 22:23:28.605000,
       2025-06-20 22:23:31.938000, 2025-06-20 22:23:35.271000,
       2025-06-20 22:23:38.604000, 2025-06-20 22:23:41.937000],
      dtype='object', length=11232)


In [11]:
src_chunks = {}

for ip in src:
    cur = start
    end = src[ip].index[-1]
    src_chunks[ip] = {}
    
    while cur < end:
        src_chunks[ip][cur] = src[ip][cur:cur+chunk_size]
        cur += chunk_size

dst_chunks = {}

for user in dst:
    cur = start
    end = dst[user].index[-1]
    dst_chunks[user] = {}
    
    while cur < end:
        dst_chunks[user][cur] = dst[user][cur:cur+chunk_size]
        cur += chunk_size

In [12]:
# - fill out zeros in the other timestamps for smaller chunks (could do this before or after chunking)
# - copy NCCTree class over
# - Send through as test (def eval(src_chunks[f], dst_chunks[f], label_func) -> List[Metrics] )
# - Setup experiment full loop (loop over all feature combos and multithread, save out to a file)

In [13]:
#dst_chunks['/tordata/config/group_0_user_0'][pd.Timestamp('2025-05-07 11:02:22.692000')]

In [14]:
def eval_model(src, src_feature, dst, dst_feature, label_func, metric_func) -> {str: float}:
    output = model(src, src_feature, dst, dst_feature)
    labels = label_func(src_chunks)
    return metric_func(output, labels)

In [15]:
import math
def ip_to_user_multi(ip, group_size=5, starting=10):
    num_isps = 10
    isp = int(int(ip.split(".")[-2]))
    node_number = (int(ip.split(".")[-1]) - starting )*num_isps + isp
    user = node_number % group_size
    group = math.floor(node_number / group_size)
    return '/tordata/config/group_' + str(group) + "_user_" + str(user)
    
def label_f(d) -> {str: str}:
    result = {}
    for k in d:
        result[k] = ip_to_user_multi(k)
    return result

print(src.keys())
print(ip_to_user_multi('102.0.8.13'))

dict_keys(['102.0.0.19', '102.0.0.10', '102.0.0.12', '102.0.0.16', '102.0.0.11', '102.0.0.18', '102.0.0.14', '102.0.0.15', '102.0.0.13', '102.0.0.17'])
/tordata/config/group_7_user_3


In [23]:
from functools import lru_cache

#@lru_cache(maxsize=None) 
def metric_match(x, y, yi):
    return  x == y[yi][0]

#@lru_cache(maxsize=None) 
def accuracy(output, labels) -> float:
    correct = 0.0
    for x in output:
        if metric_match(labels[x], output[x], 0):
            correct += 1.
    accuracy = correct/len(output)
    return accuracy

#@lru_cache(maxsize=None) 
def rank(output, labels) -> [float]:
    result = [0.]*len(output)
    
    for idx, x in enumerate(output):
        for yi in range(len(output[x])):
            if metric_match(labels[x], output[x], yi):
                result[idx] = yi + 1
                break
                
    return result

#@lru_cache(maxsize=None) 
def recall_k_f(k):
    def recall_k(output, labels) -> float:
        ranks = rank(output, labels)
        # Must remove values that are 0. These are values that could not be found
        ranks = filter(lambda x: x != 0., ranks)
        return sum([1. if r <= k else 0. for r in ranks])/len(output)
    return recall_k

#@lru_cache(maxsize=None) 
def precision_k_f(k):  
    def precision_k(output, labels) -> float:
        return recall_k_f(k)(output, labels) / float(k)
    return precision_k

#@lru_cache(maxsize=None) 
def f_beta_k_f(k, beta=1.):  
    def f_beta_k(output, labels) -> float:
        recall = recall_k_f(k)(output, labels)
        precision = precision_k_f(k)(output, labels)
        top = (1. + beta * beta) * recall * precision 
        bottom = (beta * beta * precision) + recall
        if bottom == 0:
            return 0
        return top / bottom
    return f_beta_k

#@lru_cache(maxsize=None) 
def MRR(output, labels) -> float:
    ranks = rank(output, labels)
    # Must remove values that are 0. These are values that could not be found
    ranks = filter(lambda x: x != 0., ranks)
    return sum([1./r for r in ranks])/len(output)

In [24]:
def all_metrics(output, labels):
    metrics = [
        ("Accuracy", accuracy),
        
        ("Recall@1", recall_k_f(1)),
        ("Recall@2", recall_k_f(2)),
        ("Recall@4", recall_k_f(4)),
        ("Recall@8", recall_k_f(8)),
        
        ("Precision@1", precision_k_f(1)),
        ("Precision@2", precision_k_f(2)),
        ("Precision@4", precision_k_f(4)),
        ("Precision@8", precision_k_f(8)),

        ("F-beta@1", f_beta_k_f(1)),
        ("F-beta@2", f_beta_k_f(2)),
        ("F-beta@4", f_beta_k_f(4)),
        ("F-beta@8", f_beta_k_f(8)),

        ("MRR", MRR),

        ("Rank", rank)
    ]

    return {m[0]: m[1](output, labels) for m in metrics}
    
def get_metric_names(m_f):
    return list(m_f({"": [("", 1)]}, {"": ""}).keys())


In [25]:
import sys
import os
from tqdm import tqdm
sys.path.append("./NCC")
from NCC import NCCTree

def get_ts(p, feature):
    #time = pd.Timestamp('2025-05-07 11:02:22.692000')
    #return p[time][feature].to_numpy()
    return p[feature].to_numpy()

def model(src_chunks, src_feature, dst_chunks, dst_feature, disable_bar=True) -> {str: [(str, float)]}:
    result = {}

    #noise_floor = np.mean([smallest_gt_zero(ts2nccvec(dst[user]['count'])) for user in dst])
    
    tree = NCCTree(len(src_chunks[list(src_chunks.keys())[0]]))
    for p in tqdm(dst_chunks, disable=disable_bar):
        ts = get_ts(dst_chunks[p], dst_feature)
        # advoid divide by zero if all zeros 
        if (ts == 0).all(): continue
        vec = ts2nccvec(ts)
        assert abs(1 - np.linalg.norm(vec)) < 0.001
        tree.insert(vec, p)

    for _ in range(10):
        for p in tqdm(src_chunks, disable=disable_bar):
            vec = ts2nccvec(get_ts(src_chunks[p], src_feature))
            #vec[vec < noise_floor] = 0
            n = tree.ncc(vec, 2)
            #print(ip_to_user_multi(p), n)
            result[p] = n
        
        # if in and out are set to the same then this confirms it works correctly
        # assert(all(n[i] >= n[i + 1] for i in range(len(n) - 1)))
    
    return result
model(src, 'count_ISP1-3', dst, 'count')

{'102.0.0.19': [('/tordata/config/group_18_user_0',
   np.float64(0.11653207454471382)),
  ('/tordata/config/group_15_user_1', np.float64(0.06170914978021761))],
 '102.0.0.10': [('/tordata/config/group_0_user_0',
   np.float64(0.13928987956316607)),
  ('/tordata/config/group_3_user_0', np.float64(0.0573087898320483))],
 '102.0.0.12': [('/tordata/config/group_4_user_0',
   np.float64(0.1837608709136653)),
  ('/tordata/config/group_8_user_1', np.float64(0.09105621154669852))],
 '102.0.0.16': [('/tordata/config/group_12_user_0',
   np.float64(0.158984332031268)),
  ('/tordata/config/group_12_user_3', np.float64(0.04636321884394364))],
 '102.0.0.11': [('/tordata/config/group_2_user_0',
   np.float64(0.2335463927812228)),
  ('/tordata/config/group_9_user_0', np.float64(0.0692201042482245))],
 '102.0.0.18': [('/tordata/config/group_16_user_0',
   np.float64(0.14780884952593415)),
  ('/tordata/config/group_3_user_3', np.float64(0.07652656337136464))],
 '102.0.0.14': [('/tordata/config/group_8

In [26]:
%%time
eval_model(src, 'count', dst, 'count', label_f, all_metrics)

CPU times: user 23.1 s, sys: 125 ms, total: 23.2 s
Wall time: 2.25 s


{'Accuracy': 1.0,
 'Recall@1': 1.0,
 'Recall@2': 1.0,
 'Recall@4': 1.0,
 'Recall@8': 1.0,
 'Precision@1': 1.0,
 'Precision@2': 0.5,
 'Precision@4': 0.25,
 'Precision@8': 0.125,
 'F-beta@1': 1.0,
 'F-beta@2': 0.6666666666666666,
 'F-beta@4': 0.4,
 'F-beta@8': 0.2222222222222222,
 'MRR': 1.0,
 'Rank': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [27]:
def eval_all(src, dst, label_f, metric_f):
    # WARN: This assumes all entries have the same features
    src_features = list(src.values())[0].columns
    dst_features = list(dst.values())[0].columns
    bad_features = ['frame.time']

    metric_names = get_metric_names(metric_f)

    results = pd.DataFrame(columns=['src_feature', 'dst_feature'] + metric_names)
    for df in dst_features:
        for sf in tqdm(src_features):
            if sf in bad_features: continue
            results.loc[len(results)] = {
                'src_feature': sf, 
                'dst_feature': df,
            } | eval_model(src, sf, dst, df, label_f, metric_f)
    return results

model_all_out = eval_all(src, dst, label_f, all_metrics)
model_all_out

  6%|████████████▏                                                                                                                                                                                                          | 5/88 [00:12<03:22,  2.44s/it]/tmp/ipykernel_137007/1028957165.py:5: RuntimeWarning: invalid value encountered in divide
  return (ts - ts.mean()) / (ts.std()*sample_offset)
 17%|████████████████████████████████████▍                                                                                                                                                                                 | 15/88 [00:30<02:03,  1.69s/it]/tmp/ipykernel_137007/1028957165.py:5: RuntimeWarning: invalid value encountered in divide
  return (ts - ts.mean()) / (ts.std()*sample_offset)
 40%|█████████████████████████████████████████████████████████████████████████████████████                                                                                                                        

,src_feature,dst_feature,Accuracy,Recall@1,Recall@2,Recall@4,Recall@8,Precision@1,Precision@2,Precision@4,Precision@8,F-beta@1,F-beta@2,F-beta@4,F-beta@8,MRR,Rank
0,ip.proto,text_len,1.0,1.0,1.0,1.0,1.0,1.0,0.50,0.250,0.1250,1.0,0.666667,0.40,0.222222,1.00,"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1]"
1,ip.len,text_len,1.0,1.0,1.0,1.0,1.0,1.0,0.50,0.250,0.1250,1.0,0.666667,0.40,0.222222,1.00,"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1]"
2,tcp.dstport,text_len,1.0,1.0,1.0,1.0,1.0,1.0,0.50,0.250,0.1250,1.0,0.666667,0.40,0.222222,1.00,"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1]"
3,tcp.ack,text_len,0.1,0.1,0.1,0.1,0.1,0.1,0.05,0.025,0.0125,0.1,0.066667,0.04,0.022222,0.10,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1]"
4,tcp.len,text_len,1.0,1.0,1.0,1.0,1.0,1.0,0.50,0.250,0.1250,1.0,0.666667,0.40,0.222222,1.00,"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1]"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
169,tcp.ack_Service-3,count,0.9,0.9,0.9,0.9,0.9,0.9,0.45,0.225,0.1125,0.9,0.600000,0.36,0.200000,0.90,"[1, 1, 1, 1, 0.0, 1, 1, 1, 1, 1]"
170,tcp.len_Service-3,count,1.0,1.0,1.0,1.0,1.0,1.0,0.50,0.250,0.1250,1.0,0.666667,0.40,0.222222,1.00,"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1]"
171,tcp.time_relative_Service-3,count,0.7,0.7,0.8,0.8,0.8,0.7,0.40,0.200,0.1000,0.7,0.533333,0.32,0.177778,0.75,"[1, 2, 0.0, 1, 0.0, 1, 1, 1, 1, 1]"
172,tcp.time_delta_Service-3,count,0.7,0.7,0.8,0.8,0.8,0.7,0.40,0.200,0.1000,0.7,0.533333,0.32,0.177778,0.75,"[1, 2, 0.0, 1, 0.0, 1, 1, 1, 1, 1]"


In [29]:
model_all_out.sort_values(by='Accuracy', ascending=False).head(40)

,src_feature,dst_feature,Accuracy,Recall@1,Recall@2,Recall@4,Recall@8,Precision@1,Precision@2,Precision@4,Precision@8,F-beta@1,F-beta@2,F-beta@4,F-beta@8,MRR,Rank
0,ip.proto,text_len,1.0,1.0,1.0,1.0,1.0,1.0,0.5,0.25,0.125,1.0,0.666667,0.4,0.222222,1.0,"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1]"
1,ip.len,text_len,1.0,1.0,1.0,1.0,1.0,1.0,0.5,0.25,0.125,1.0,0.666667,0.4,0.222222,1.0,"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1]"
2,tcp.dstport,text_len,1.0,1.0,1.0,1.0,1.0,1.0,0.5,0.25,0.125,1.0,0.666667,0.4,0.222222,1.0,"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1]"
4,tcp.len,text_len,1.0,1.0,1.0,1.0,1.0,1.0,0.5,0.25,0.125,1.0,0.666667,0.4,0.222222,1.0,"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1]"
9,count,text_len,1.0,1.0,1.0,1.0,1.0,1.0,0.5,0.25,0.125,1.0,0.666667,0.4,0.222222,1.0,"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1]"
6,udp.dstport,text_len,1.0,1.0,1.0,1.0,1.0,1.0,0.5,0.25,0.125,1.0,0.666667,0.4,0.222222,1.0,"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1]"
12,tcp.dstport_ISP1-all,text_len,1.0,1.0,1.0,1.0,1.0,1.0,0.5,0.25,0.125,1.0,0.666667,0.4,0.222222,1.0,"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1]"
13,tcp.ack_ISP1-all,text_len,1.0,1.0,1.0,1.0,1.0,1.0,0.5,0.25,0.125,1.0,0.666667,0.4,0.222222,1.0,"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1]"
11,ip.len_ISP1-all,text_len,1.0,1.0,1.0,1.0,1.0,1.0,0.5,0.25,0.125,1.0,0.666667,0.4,0.222222,1.0,"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1]"
10,ip.proto_ISP1-all,text_len,1.0,1.0,1.0,1.0,1.0,1.0,0.5,0.25,0.125,1.0,0.666667,0.4,0.222222,1.0,"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1]"


In [ ]:
def best_scopes(results):
    

In [ ]:
def smallest_gt_zero(df):
    return min(df[df > 0])
#smallest_gt_zero(y2)

In [ ]:
import matplotlib.pyplot as plt
import datetime

noise_floor = np.mean([smallest_gt_zero(ts2nccvec(dst[user]['count'])) for user in dst])

x = src['102.0.0.10'].index
y = ts2nccvec(src['102.0.0.10']['count'])
y2 = ts2nccvec(dst['/tordata/config/group_0_user_0']['count'])

y.loc[y < noise_floor] = 0


plt.plot(x,y)
plt.plot(x, y2)
plt.show()
np.dot(y.to_numpy(), y2.to_numpy())#/np.dot(y.to_numpy(), y.to_numpy())

In [ ]:
pd.unique(y2)
y2

In [ ]:
np.min([smallest_gt_zero(ts2nccvec(dst[user]['count'])) for user in dst])

In [ ]:
list(src['102.0.0.10'].columns)

In [ ]:
def model_OLD(src_raw, src_feature, dst_raw, dst_feature):
    import statsmodels.api as sm
    def ccf_calc(sig1, sig2):
        corr = sm.tsa.stattools.ccf(sig2, sig1, adjusted=False, nlags=1)
    
        # Remove padding and reverse the order
        return corr[0:(len(sig2)+1)][::-1]


    def cross_cor(ts1, ts2, debug=False, max_offset=300, only_positive=True):
        ccf = ccf_calc(ts1, ts2)
        best_cor = max(ccf)
        best_lag = np.argmax(ccf)
    
        if debug:
            print('best cross correlation: ' + str(best_cor) + " at time lag: " + str(best_lag))
            print(len(ccf))
            print(ccf)
            ccf_plot(range(len(ccf)), ccf)
        return best_cor, best_lag

    def compare_ts(ts1, ts2, debug=False):
        # dtw_classic, path_classic = dtw(ts1, ts2, dist='square',
        #                             method='classic', return_path=True)
        # return dtw_classic
        # print(ts1)
        # print(ts2)
        # dist, lag = cross_cor(pd.Series(ts1), pd.Series(ts2))
        dist, lag = cross_cor(ts1, ts2, debug=debug)
        # assert dist >= -1 and dist <= 1
        dist = dist * -1  # flip for use as distance metric
        # assert dist >= -1 and dist <= 1
        return dist, lag

    def compare_ts_reshape(ts1, ts2, debug=False):
        ts1 = ts1.fillna(0)
        ts2 = ts2.fillna(0)
    
        range = min(ts2.index.values), max(ts2.index.values)
        ts1 = ts1.loc[(ts1.index >= range[0]) & (ts1.index <= range[1])]
    
        # ts1 = ts1.loc[:, 'tda_pl']
        ts1 = ts1.values
    
        ts1_norm = np.array(ts1)
        ts2_norm = np.array(ts2)
    
        # delay = 0
    
        # ts1_norm.index = ts1_norm.index + pd.DateOffset(seconds=delay)
    
        # lock to same range with buffer room
        # on each side to account for network (or PPT) delay
    
        # detect if no overlap
        if len(ts1_norm) < 2 or len(ts2_norm) < 2:
            return float("inf"), 0
    
        # Normalize peaks?
        # ts1_norm = normalize_ts(ts1_norm)
        # ts2_norm = normalize_ts(ts2_norm)
    
        # plot_ts(ts1_norm, ts2_norm)
        # exit(1)
    
        # else:
        #     ts1_norm = ts1_norm.tolist()
        #     ts2_norm = ts2_norm.tolist()
    
        score, lag = compare_ts(ts1_norm, ts2_norm, debug=debug)
        # adj_score = score + (((lag+1)**1.1)/1000.0)
        return score, lag

    src = {}
    dst = {}
    for ip in src_raw:
        src[ip] = src_raw[ip][src_feature]
    for user in dst_raw:
        dst[user] = dst_raw[user][dst_feature]
        
    correct = 0.0
    for user in dst:
        best_score = 0
        best_user = 0
        for _ in range(10):
            for ip in src:
                score = sm.tsa.stattools.ccf(src[ip].values, dst[user].values, adjusted=False, nlags=1)
                if score > best_score:
                    best_score = score
                    best_user = ip_to_user_multi(ip)
        if user == best_user:
            correct += 1
    accuracy = correct / len(src)
    return accuracy

In [ ]:
%%time
model_OLD(src, 'count', dst, 'count')